# SAHPRA guidelines Q&A — RAG demo with Qwen on Colab

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.
Then run the cells from top to bottom (Runtime → Run all works too).

Steps:
1. Install Ollama and download Qwen (about 6 GB, takes a few minutes)
2. Check Qwen answers a question
3. Load `passages.jsonl` from the preprocessing script
4. Embed the passages and search them
5. Ask questions: retrieve passages → Qwen answers with citations

## 1. Install Ollama and start it

In [ ]:
# zstd is needed by the Ollama installer; pciutils lets it detect the GPU
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests

# Start the Ollama server in the background
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):                     # wait until it's ready
    try:
        requests.get("http://localhost:11434", timeout=1)
        print("Ollama is running")
        break
    except requests.ConnectionError:
        time.sleep(1)

## 2. Download Qwen and test it

`qwen3.5:9b` is a 4-bit version (about 6 GB) that fits easily on the T4.
If the download says the model isn't found, check the exact tag at https://ollama.com/library/qwen3.5

In [ ]:
MODEL = "qwen3.5:9b"
!ollama pull {MODEL}

In [ ]:
def ask_qwen(messages):
    """Send a chat to Qwen and return its reply."""
    r = requests.post("http://localhost:11434/api/chat", json={
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "think": False,                       # skip the long reasoning section
        "options": {"temperature": 0.2},      # low = more factual, less creative
    }, timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]

print(ask_qwen([{"role": "user", "content": "In one sentence, what does SAHPRA regulate?"}]))

## 3. Load the passages

Put `passages.jsonl` (made by `preprocess_sahpra.py`) in a Google Drive folder called `sahpra`.
Alternatively, set `USE_DRIVE = False` to upload the file directly.

In [ ]:
import json

USE_DRIVE = True
PATH = "/content/drive/MyDrive/sahpra/passages.jsonl"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    from google.colab import files
    uploaded = files.upload()               # choose passages.jsonl
    PATH = next(iter(uploaded))

with open(PATH, encoding="utf-8") as f:
    passages = [json.loads(line) for line in f]

print(f"Loaded {len(passages)} passages from {len({p['source_file'] for p in passages})} documents")

## 4. Embed the passages

An embedding model turns each passage into a list of numbers (a vector).
Passages with similar meaning get similar vectors, so a question can be matched to the passages that answer it.

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")

texts = [p["text"] for p in passages]
passage_vectors = embedder.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print("Vectors:", passage_vectors.shape)   # (number of passages, 384)

In [ ]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # recommended for BGE models

def search(question, k=5):
    """Return the k passages most similar to the question."""
    q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True)[0]
    scores = passage_vectors @ q             # cosine similarity (vectors are normalised)
    best = np.argsort(-scores)[:k]
    return [(passages[i], float(scores[i])) for i in best]

for p, score in search("How do I report an adverse drug reaction?", k=3):
    print(f"{score:.3f}  {p['document_number']}  {p['title'][:60]}  (pages {p['page_start']}-{p['page_end']})")

## 5. Ask questions

In [ ]:
SYSTEM_PROMPT = (
    "You answer questions about South African Health Products Regulatory Authority (SAHPRA) guidelines. "
    "Use ONLY the numbered sources provided. Cite sources in square brackets, e.g. [1] or [2][3]. "
    "If the sources do not contain the answer, say you could not find it in the guidelines."
)

def source_label(p):
    pages = f", p. {p['page_start']}" if p["page_start"] == p["page_end"] and p["page_start"] else \
            f", pp. {p['page_start']}-{p['page_end']}" if p["page_start"] else ""
    return f"{p['document_number'] or p['source_file']}: {p['title']}{pages}"

def answer(question, k=5):
    results = search(question, k)
    context = "\n\n".join(f"[{i}] {source_label(p)}\n{p['text']}" for i, (p, _) in enumerate(results, start=1))

    reply = ask_qwen([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Sources:\n{context}\n\nQuestion: {question}"},
    ])

    print("QUESTION:", question, "\n")
    print(reply, "\n")
    print("SOURCES:")
    for i, (p, score) in enumerate(results, start=1):
        print(f"  [{i}] {source_label(p)}  (similarity {score:.2f})")

In [ ]:
answer("What must be included when submitting a variation application?")

In [ ]:
# Try your own questions here
answer("How should a reference product be selected for a generic medicine?")